# Machine Learning 2025W — Exercise 0  
### Dataset Description and Exploration (adult)

**Group Members:**  
- Benjamin Weber  
- Donner Adrian
- Florian Hess 

---


## 1. Setup

In [ ]:
from scipy.io import arff
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

file_path = '../datasets/dataset_adult.arff'

# Read ARFF file, skip lines starting with '%' or '@'
with open(file_path, 'r') as f:
    lines = [line.strip() for line in f if not line.startswith('@') and not line.startswith('%') and line.strip() != '']

# Split each line by comma
data = [line.split(',') for line in lines]

# Column names from ARFF attributes
columns = ['age','fnlwgt','education-num','capital-gain','capital-loss','hours-per-week',
           'workclass','education','marital-status','occupation','relationship',
           'race','sex','native-country','class']

# Create DataFrame
df = pd.DataFrame(data, columns=columns)

# Convert numeric columns
numeric_cols = ['age','fnlwgt','education-num','capital-gain','capital-loss','hours-per-week']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

# Convert categorical columns
categorical_cols = [c for c in df.columns if c not in numeric_cols]
df[categorical_cols] = df[categorical_cols].apply(lambda x: x.astype('category'))

df.head(20)

## 2. Load old Dataset

In [ ]:
file_path = '../datasets/dataset_adult.arff'

data, meta = arff.loadarff(file_path)
df_old = pd.DataFrame(data)

for col in df_old.select_dtypes(['object']).columns:
        df_old[col] = df_old[col].str.decode('utf-8')

print("Dataset shape:", df_old.shape)
df_old.head(20)

## 3. Basic Information

In [ ]:
df.info()
df.describe()
df.isna().sum()


In [ ]:
df['workclass'].value_counts(dropna=False)
df['occupation'].value_counts(dropna=False)

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# 1) Basic counts
n_samples, n_attrs = df.shape
print(f"Samples: {n_samples:,}, Attributes: {n_attrs:,}")

# 2) Per-column diagnostics
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_missing': df.isna().sum(),
    'n_unique': df.nunique(dropna=False)
})

# 3) Heuristic to infer attribute type
def infer_attr_type(col):
    if pd.api.types.is_numeric_dtype(df[col]):
        # numeric: if integer with few unique values -> maybe ordinal; else interval/ratio
        if df[col].nunique(dropna=True) <= 10 and pd.api.types.is_integer_dtype(df[col]):
            return 'ordinal (likely)'
        return 'interval/ratio'
    else:
        # non-numeric -> categorical (nominal)
        return 'nominal (categorical)'

summary['inferred_type'] = [infer_attr_type(c) for c in df.columns]

# show columns sorted by number of missing values (descending)
display(summary.sort_values('n_missing', ascending=False))

## 4. Target Variable (Distribution)

In [ ]:
target_col = 'class'

sns.countplot(x=target_col, data=df)
plt.title('Target distribution (Income <=50K vs >50K)')
plt.xlabel('Income Class (class)')

## 5. Feature Exploration

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols].hist(figsize=(12, 8))
plt.suptitle("Numeric Feature Distributions")
plt.show()

cat_cols = df.select_dtypes(exclude=np.number).columns
for col in cat_cols:
    plt.figure(figsize=(8, 3))
    sns.countplot(x=df[col])
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
sns.countplot(y='occupation', data=df, order=df['occupation'].value_counts().index)
plt.title('Occupation frequencies')
plt.show()

In [ ]:
df['target_encoded'] = (df['class'] == '>50K').astype(int)
corr = df.corr(numeric_only=True)['target_encoded'].sort_values(ascending=False)
corr